# Retail Customer Behavior Analysis

**Author: Hamzah Al-Hakem | Python + Power BI | Portfolio case study**

This notebook prepares and explores a 3,900-record synthetic retail sample. It preserves original review ratings, creates explicit missingness and imputed fields, and builds age bands and purchase-history quartiles.

**Business question:** Where is recorded purchase value concentrated, and which promotion and subscription hypotheses deserve controlled testing?

## Before running

Read `data/README.md` and supply an authorized `data/raw/customer_shopping_behavior.csv`. The data is intentionally absent from the public repository. The audited project variant has 37 missing ratings; the upstream download has none, so rating-related results may differ. Run from the repository root or the `notebooks` folder. The final CSV is written to `data/processed/` and is ignored by Git.

Saved outputs are cleared to avoid redistributing row-level data. Verified aggregate findings are in the README, report and validation files. No profit, retention, lifetime value or causal marketing effect is estimated here.


## 1. Load the authorized source

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "raw" / "customer_shopping_behavior.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError("Supply an authorized CSV at data/raw/customer_shopping_behavior.csv; see data/README.md.")
df = pd.read_csv(DATA_PATH)


In [ ]:
df.shape

In [ ]:
df.head(5)

## 2. Inspect data quality

In [ ]:
df.duplicated().sum()

In [ ]:
total_revenue = df['Purchase Amount (USD)'].sum()
print(f"Total revenue = ${total_revenue:,.2f}")

In [ ]:
df.isnull().sum()

In [ ]:
df.groupby('Category')['Review Rating'].agg(
    missing_count = lambda x: x.isnull().sum(),
    median_rating = 'median'
)


### Preserve observed and filled ratings separately

In [ ]:
df['Rating Was Missing'] = df['Review Rating'].isna()

In [ ]:
df.info()

In [ ]:
df['Review Rating Filled'] = (
    df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))
)

In [ ]:
df.loc[
    df['Rating Was Missing'],
    ['Customer ID', 'Category', 'Review Rating',
     'Review Rating Filled']
].head(20)

In [ ]:
df[['Review Rating', 'Review Rating Filled']].isna().sum()

In [ ]:
mask = df['Review Rating'].notna()

(
    df.loc[mask, 'Review Rating']
    == df.loc[mask, 'Review Rating Filled']
).all()

### Standardize field names

In [ ]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'[^a-z0-9]+', '_', regex=True)
      .str.strip('_')
)

In [ ]:
print(df.columns.tolist())

In [ ]:
print(df.columns.is_unique)

In [ ]:
print(df.shape)

### Check numeric ranges and customer grain

In [ ]:
df[[
    'age',
    'purchase_amount_usd',
    'review_rating',
    'previous_purchases'
]].describe()

In [ ]:
assert (df['purchase_amount_usd'] > 0).all()

assert df['review_rating'].dropna().between(1, 5).all()

assert (df['age'] > 0).all()

assert df['customer_id'].is_unique

print("All validation checks passed!")

In [ ]:
df['frequency_of_purchases'].value_counts()

## 3. Engineer interpretable segments

In [ ]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[17, 29, 44, 59, float('inf')],
    labels=['18-29', '30-44', '45-59', '60+']
)

In [ ]:
df['age_group'].value_counts().sort_index()

In [ ]:
df['purchase_history_segment'] = pd.qcut(
    df['previous_purchases'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

In [ ]:
df.groupby(
    'purchase_history_segment',
    observed=True
)['previous_purchases'].agg(['min', 'max', 'count'])

## 4. Compare customer segments

Revenue totals reflect both segment size and average recorded purchase. Q1-Q4 denote purchase-history quartiles, not calendar quarters or proven loyalty tiers.

In [ ]:
gender_analysis = df.groupby('gender').agg(
    customers=('customer_id', 'count'),
    total_revenue=('purchase_amount_usd', 'sum'),
    avg_spend=('purchase_amount_usd', 'mean')
).round(2)

print(gender_analysis)

In [ ]:
age_analysis = df.groupby(
    'age_group', observed=True
).agg(
    customers=('customer_id', 'count'),
    total_revenue=('purchase_amount_usd', 'sum'),
    avg_spend=('purchase_amount_usd', 'mean')
).round(2)

print(age_analysis)

In [ ]:
gender_revenue = (
    df.groupby('gender')['purchase_amount_usd']
      .sum()
      .sort_values(ascending=False)
)

print(gender_revenue)

In [ ]:
gender_analysis = (
    df.groupby('gender')
      .agg(
          customers=('customer_id', 'nunique'),
          total_revenue=('purchase_amount_usd', 'sum'),
          avg_purchase=('purchase_amount_usd', 'mean')
      )
      .round(2)
)

gender_analysis

In [ ]:
age_analysis = (
    df.groupby('age_group', observed=True)
      .agg(
          customers=('customer_id', 'nunique'),
          total_revenue=('purchase_amount_usd', 'sum'),
          avg_purchase=('purchase_amount_usd', 'mean')
      )
      .round(2)
)

age_analysis

In [ ]:
segment_analysis = (
    df.groupby('purchase_history_segment', observed=True)
      .agg(
          customers=('customer_id', 'nunique'),
          avg_purchase=('purchase_amount_usd', 'mean')
      )
      .round(2)
)

segment_analysis

## 5. Explore product and seasonal contribution

In [ ]:
category_analysis = (
    df.groupby('category')
      .agg(
          purchases=('customer_id', 'count'),
          revenue=('purchase_amount_usd', 'sum'),
          avg_purchase=('purchase_amount_usd', 'mean')
      )
      .round(2)
)

category_analysis['revenue_share_pct'] = (
    category_analysis['revenue']
    / category_analysis['revenue'].sum() * 100
).round(2)

category_analysis = category_analysis.sort_values(
    'revenue', ascending=False
)

category_analysis

In [ ]:
season_analysis = (
    df.groupby('season')
      .agg(
          purchases=('customer_id', 'count'),
          revenue=('purchase_amount_usd', 'sum'),
          avg_purchase=('purchase_amount_usd', 'mean')
      )
      .round(2)
      .sort_values('revenue', ascending=False)
)

season_analysis

In [ ]:
top_products = (
    df.groupby('item_purchased')
      .agg(
          purchases=('customer_id', 'count'),
          revenue=('purchase_amount_usd', 'sum')
      )
      .sort_values('revenue', ascending=False)
      .head(5)
)

top_products

In [ ]:
import matplotlib.pyplot as plt

top_products.sort_values('revenue').plot(
    kind='barh',
    y='revenue',
    legend=False,
    figsize=(8, 4)
)

plt.title('Top 5 Products by Revenue')
plt.xlabel('Revenue (USD)')
plt.ylabel('Product')
plt.tight_layout()
plt.show()

## 6. Examine promotions and observed ratings

Ratings use only nonmissing original values. Discount comparisons describe associations; they do not estimate incremental campaign impact.

In [ ]:
discount_analysis = (
    df.groupby('discount_applied')
      .agg(
          purchases=('customer_id', 'count'),
          revenue=('purchase_amount_usd', 'sum'),
          avg_spend=('purchase_amount_usd', 'mean')
      )
      .round(2)
)

discount_analysis

In [ ]:
category_ratings = (
    df.groupby('category')
      .agg(
          total_ratings=('review_rating', 'count'),
          avg_rating=('review_rating', 'mean')
      )
      .round(2)
      .sort_values('avg_rating', ascending=False)
)

category_ratings

In [ ]:
product_ratings = (
    df.groupby('item_purchased')
      .agg(
          avg_rating=('review_rating', 'mean'),
          rating_count=('review_rating', 'count')
      )
)

top_rated_products = (
    product_ratings[
        product_ratings['rating_count'] >= 30
    ]
    .sort_values('avg_rating', ascending=False)
    .head(5)
    .round(2)
)

top_rated_products

## 7. Evaluate subscription patterns

In [ ]:
loyalty_analysis = (
    df.assign(
        is_subscribed=df['subscription_status'].eq('Yes')
    )
    .groupby('purchase_history_segment', observed=True)
    .agg(
        customers=('customer_id', 'count'),
        subscribers=('is_subscribed', 'sum'),
        subscription_rate=('is_subscribed', 'mean')
    )
)

loyalty_analysis['subscription_rate'] = (
    loyalty_analysis['subscription_rate'] * 100
).round(2)

loyalty_analysis

In [ ]:
subscription_analysis = (
    df.groupby('subscription_status')
      .agg(
          customers=('customer_id', 'count'),
          revenue=('purchase_amount_usd', 'sum'),
          avg_purchase=('purchase_amount_usd', 'mean')
      )
      .round(2)
)

subscription_analysis

In [ ]:
pd.crosstab(
    df['subscription_status'],
    df['discount_applied']
)

## 8. Validate and export

In [ ]:
# Required columns
required_cols = [
    'customer_id',
    'purchase_amount_usd',
    'review_rating',
    'review_rating_filled',
    'rating_was_missing',
    'age_group',
    'purchase_history_segment'
]

assert set(required_cols).issubset(df.columns)

# Validate data
assert len(df) == 3900
assert df['customer_id'].is_unique
assert df['purchase_amount_usd'].gt(0).all()

assert (
    df['rating_was_missing']
    == df['review_rating'].isna()
).all()

assert df['review_rating_filled'].notna().all()

print("Data validation passed!")

In [ ]:
from pathlib import Path

# Create output directory
output_dir = ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Export cleaned dataset
file_path = output_dir / "customer_shopping_cleaned.csv"

df.to_csv(file_path, index=False)

print("Cleaned dataset exported successfully!")

In [ ]:
df_check = pd.read_csv(file_path)

print("Rows:", df_check.shape[0])
print("Columns:", df_check.shape[1])
print("Missing Ratings:", df_check['review_rating'].isna().sum())

## 9. Interpret the evidence

- Clothing is the largest recorded revenue pool: $104,264 (44.73%) in the audited snapshot.
- Ages 45-59 have the highest total recorded revenue, partly because this is the largest age group. Average purchase values are similar across age groups.
- Q3 has the highest subscription rate (29.20%); the rate does not rise consistently across purchase-history quartiles.
- All 1,053 subscribers received discounts. The missing subscriber-without-discount comparison prevents separation of these effects.
- Test incremental contribution margin and conversion using randomized campaign assignment before increasing promotional spending.

These are descriptive results from synthetic data, not real-world forecasts. See `docs/AUDIT.md` for verification scope and Power BI changes.
